<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/APAC_AGENTIC_GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/frank-morales2020/AST/blob/main/Qwen3dot8_27B_topo_cbp_gguf.ipynb

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/*.gguf

In [ ]:
!apt-get update && !apt-get install -y git
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --force-reinstall --no-cache-dir git+https://github.com/abetlen/llama-cpp-python.git --verbose

In [1]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


GPU Offload Supported: True


In [2]:
import os
import struct
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/GGUF_Models")

def read_gguf_header(filepath):
    try:
        with open(filepath, "rb") as f:
            magic = f.read(4)
            if magic != b"GGUF":
                return {"error": "Invalid magic bytes"}
            version = struct.unpack("<I", f.read(4))[0]
            tensor_count = struct.unpack("<Q", f.read(8))[0]
            kv_count = struct.unpack("<Q", f.read(8))[0]
            return {
                "version": version,
                "tensor_count": tensor_count,
                "kv_count": kv_count
            }
    except Exception as e:
        return {"error": str(e)}

print(f"{'Model Filename':<52} | {'Size (GB)':<10} | {'GGUF Version':<12} | {'Status'}")
print("-" * 95)

total_size = 0
for file_path in sorted(model_dir.glob("*.gguf")):
    file_size = file_path.stat().st_size
    size_gb = file_size / (1024**3)
    total_size += file_size

    meta = read_gguf_header(file_path)
    if "error" not in meta:
        status = "Verified"
        version_str = f"v{meta['version']}"
    else:
        status = meta["error"]
        version_str = "N/A"

    print(f"{file_path.name:<52} | {size_gb:>8.2f}GB | {version_str:<12} | {status}")

print("-" * 95)
print(f"Total Portfolio Storage: {total_size / (1024**3):.2f} GB across {len(list(model_dir.glob('*.gguf')))} models.")

Model Filename                                       | Size (GB)  | GGUF Version | Status
-----------------------------------------------------------------------------------------------
certified-scorpion-topo-qwen-bf16.gguf               |    14.19GB | v3           | Verified
deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf |    69.83GB | v3           | Verified
deepseek_moe_16b_topocbp_q4_k_m.gguf                 |    10.11GB | v3           | Verified
gpt-oss-20b-topo-governed-Q4_K_M.gguf                |    14.72GB | v3           | Verified
llama-3.1-70b-topo-governed-q4_k_m.gguf              |    39.60GB | v3           | Verified
mixtral_8x22b_q4_k_m.gguf                            |    79.71GB | v3           | Verified
muse-glimmer-30b-card-faithful-Q4_K_M.gguf           |    15.77GB | v3           | Verified
nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf          |     4.97GB | v3           | Verified
qwen3.8-27b-topocbp-Q8_0.gguf                        |    26.63GB | v3       

In [1]:
import os
from llama_cpp import Llama

class HybridAPACAgentSystem:
    def __init__(self, model_dir="/content/drive/MyDrive/GGUF_Models"):
        print("Initializing Hybrid APAC Agent Ecosystem (95GB VRAM Custom Offloading)...")

        # 1. Router: DeepSeek MoE (Smallest ~10.11 GB) -> Full GPU Offload
        print("Loading Router: DeepSeek MoE (n_gpu_layers=-1)...")
        self.router = Llama(
            model_path=os.path.join(model_dir, "deepseek_moe_16b_topocbp_q4_k_m.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 2. Synthesizer: Qwen 3.8-27B (Medium ~26.63 GB) -> Full GPU Offload
        print("Loading Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...")
        self.synthesizer = Llama(
            model_path=os.path.join(model_dir, "qwen3.8-27b-topocbp-Q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 3. Multilingual Specialist: Sarvam-30B (~31.86 GB) -> Full GPU Offload
        print("Loading Multilingual: Sarvam-30B (n_gpu_layers=25)...")
        self.multilingual = Llama(
            model_path=os.path.join(model_dir, "sarvam-30b-topo-governed-q8_0.gguf"),
            n_ctx=4096, n_gpu_layers=25, verbose=False
        )

        # 4. Foundational: Certified Scorpion Qwen (~14.19 GB) -> Full GPU Offload
        print("Loading Foundational: Certified Scorpion Qwen (n_gpu_layers=-1)...")
        self.foundational = Llama(
            model_path=os.path.join(model_dir, "certified-scorpion-topo-qwen-bf16.gguf"),
            n_ctx=4096, n_gpu_layers=-1, verbose=False
        )

        # 5. Deep Reasoner: DeepSeek-R1 70B (Biggest ~69.83 GB) -> Partial Offload (n_gpu_layers=35)
        print("Loading Deep Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...")
        self.reasoner = Llama(
            model_path=os.path.join(model_dir, "deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"),
            n_ctx=8192, n_gpu_layers=25, verbose=False
        )

        print("All 5 APAC models active with custom offloading profiles under Topo-CBP governance.")

    def route_task(self, query: str) -> str:
        prompt = (
            f"Classify the following query into one of five categories "
            f"[REASONING, SYNTHESIS, MULTILINGUAL, FOUNDATIONAL, GENERAL]:\n\n{query}\nCategory:"
        )
        res = self.router(prompt, max_tokens=20, temperature=0.1)
        category = res["choices"][0]["text"].strip().upper()
        for cat in ["REASONING", "SYNTHESIS", "MULTILINGUAL", "FOUNDATIONAL", "GENERAL"]:
            if cat in category:
                return cat
        return "SYNTHESIS"

    def execute_pipeline(self, query: str):
        task_type = self.route_task(query)
        print(f"\n[Router: DeepSeek MoE] Task classified as: {task_type}")

        if task_type == "REASONING":
            print("[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep reasoning...")
            prompt = f"<|im_start|>system\nYou are a rigorous logical reasoning agent.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n<think>\n"
            res = self.reasoner(prompt, max_tokens=1536, temperature=0.5, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "SYNTHESIS":
            print("[Agent: Qwen 3.8-27B (n_gpu_layers=25] Executing synthesis...")
            prompt = f"<|im_start|>system\nYou are an expert synthesis agent.<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.6, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        elif task_type == "MULTILINGUAL":
            print("[Agent: Sarvam-30B (n_gpu_layers=25)] Processing regional context...")
            prompt = f"System: You are an expert multilingual processor.\nUser: {query}\nAssistant:"
            res = self.multilingual(prompt, max_tokens=1024, temperature=0.7)
            return res["choices"][0]["text"].strip()

        elif task_type == "FOUNDATIONAL":
            print("[Agent: Certified Scorpion Qwen (n_gpu_layers=-1)] Applying instruction tuning...")
            prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.foundational(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

        else:
            print("[Agent: Qwen Synthesizer] Handling general objective...")
            prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
            res = self.synthesizer(prompt, max_tokens=1024, temperature=0.7, stop=["<|im_end|>"])
            return res["choices"][0]["text"].strip()

# --- Execution Test ---
if __name__ == "__main__":
    system = HybridAPACAgentSystem()
    sample_query = "Prove the convergence of continual backpropagation under topological invariant locking."
    result = system.execute_pipeline(sample_query)

    print("\n==============================")
    print("FINAL HYBRID PIPELINE OUTPUT")
    print("==============================\n")
    print(result)

Initializing Hybrid APAC Agent Ecosystem (95GB VRAM Custom Offloading)...
Loading Router: DeepSeek MoE (n_gpu_layers=-1)...
Loading Synthesizer: Qwen 3.8-27B (n_gpu_layers=25)...
Loading Multilingual: Sarvam-30B (n_gpu_layers=25)...
Loading Foundational: Certified Scorpion Qwen (n_gpu_layers=-1)...
Loading Deep Reasoner: DeepSeek-R1 70B (n_gpu_layers=25)...
All 5 APAC models active with custom offloading profiles under Topo-CBP governance.

[Router: DeepSeek MoE] Task classified as: REASONING
[Agent: DeepSeek-R1 70B (n_gpu_layers=25)] Executing deep reasoning...

FINAL HYBRID PIPELINE OUTPUT

Okay, so I need to prove the convergence of continual backpropagation under topological invariant locking. Hmm, I'm not exactly sure what that means, but I'll try to break it down. 

First, I know that backpropagation is a common algorithm used in training neural networks. It's about adjusting the weights to minimize the error. Continual backpropagation might refer to updating the weights continuo

In [2]:
!nvidia-smi

Sun Nov 15 00:21:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             88W /  600W |   92929MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----